<!--TITLE: Maximum Pooling-->

# Введение #

В Уроке 2 мы начали обсуждение того, как базовая часть свёрточной сети (convnet) выполняет извлечение признаков. Мы узнали, как первые две операции этого процесса происходят в слое `Conv2D` с активацией `relu`.

В этом уроке мы рассмотрим третью (и последнюю) операцию в этой последовательности: **уплотнение (condense)** с помощью **максимального пулинга (maximum pooling)**, который в Keras реализуется слоем `MaxPool2D`.

# Уплотнение с помощью максимального пулинга #

Добавление шага уплотнения в модель, которая у нас была ранее, даст нам следующее:

```python
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Conv2D(filters=64, kernel_size=3), # активация отсутствует
    layers.MaxPool2D(pool_size=2),
    # Далее следуют другие слои
])
```

Слой `MaxPool2D` во многом похож на слой `Conv2D`, за исключением того, что он использует простую функцию максимума вместо ядра (kernel), а параметр `pool_size` аналогичен `kernel_size`. Однако, в отличие от свёрточного слоя, слой `MaxPool2D` не имеет обучаемых весов в своём ядре.

Давайте ещё раз взглянем на рисунок извлечения признаков из прошлого урока. Помните, что `MaxPool2D` — это шаг **Уплотнение (Condense)**.

<figure>
<img src="./img/IYO9lqp.png" width="600" alt="Пример процесса извлечения признаков.">
</figure>

Обратите внимание, что после применения функции ReLU (**Обнаружение (Detect)**) карта признаков содержит много «мёртвого пространства» — больших областей, заполненных только нулями (чёрные области на изображении). Перенос этих нулевых активаций через всю сеть увеличил бы размер модели, не добавляя полезной информации. Вместо этого мы хотим *уплотнить* карту признаков, сохранив только самую полезную часть — сам признак.

Именно это и делает **максимальный пулинг**. Максимальный пулинг берёт патч активаций из исходной карты признаков и заменяет их максимальной активацией в этом патче.

<figure>
<img src="./img/hK5U2cd.png" width="400" alt="Максимальный пулинг заменяет патч максимальным значением в этом патче.">
</figure>

При применении после активации ReLU это даёт эффект «усиления» признаков. Шаг пулинга увеличивает долю активных пикселей по отношению к нулевым.

# Пример — Применение максимального пулинга #

Давайте добавим шаг «уплотнения» к извлечению признаков, которое мы делали в примере из Урока 2. Следующая скрытая ячейка вернёт нас к тому месту, где мы остановились.

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import warnings

plt.rc('figure', autolayout=True)
plt.rc('axes', labelweight='bold', labelsize='large',
       titleweight='bold', titlesize=18, titlepad=10)
plt.rc('image', cmap='magma')
warnings.filterwarnings("ignore") # для чистоты выходных ячеек

# Чтение изображения
image_path = './data/car_feature.jpg'
image = tf.io.read_file(image_path)
image = tf.io.decode_jpeg(image)

# Определение ядра
kernel = tf.constant([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1],
], dtype=tf.float32)

# Переформатирование для совместимости с батчами
image = tf.image.convert_image_dtype(image, dtype=tf.float32)
image = tf.expand_dims(image, axis=0)
kernel = tf.reshape(kernel, [*kernel.shape, 1, 1])

# Шаг фильтрации
image_filter = tf.nn.conv2d(
    input=image,
    filters=kernel,
    # об этих двух параметрах мы поговорим в следующем уроке!
    strides=1,
    padding='SAME'
)

# Шаг обнаружения
image_detect = tf.nn.relu(image_filter)

# Покажем, что у нас получилось
plt.figure(figsize=(12, 6))
plt.subplot(131)
plt.imshow(tf.squeeze(image), cmap='gray')
plt.axis('off')
plt.title('Вход')
plt.subplot(132)
plt.imshow(tf.squeeze(image_filter))
plt.axis('off')
plt.title('Фильтр')
plt.subplot(133)
plt.imshow(tf.squeeze(image_detect))
plt.axis('off')
plt.title('Обнаружение')
plt.show();

Мы используем ещё одну функцию из `tf.nn` для применения шага пулинга — `tf.nn.pool`. Это функция Python, которая делает то же самое, что и слой `MaxPool2D`, используемый при построении модели, но, будучи простой функцией, её проще использовать напрямую.

In [ ]:
import tensorflow as tf

image_condense = tf.nn.pool(
    input=image_detect, # изображение после шага Обнаружение выше
    window_shape=(2, 2),
    pooling_type='MAX',
    # мы увидим, что это делает, в следующем уроке!
    strides=(2, 2),
    padding='SAME',
)

plt.figure(figsize=(6, 6))
plt.imshow(tf.squeeze(image_condense))
plt.axis('off')
plt.show();

Довольно круто! Надеюсь, вы видите, как шаг пулинга смог усилить признак, уплотнив изображение вокруг наиболее активных пикселей.

# Инвариантность к сдвигу (Translation Invariance) #

Мы назвали нулевые пиксели «неважными». Означает ли это, что они вообще не несут информации? На самом деле нулевые пиксели несут *позиционную информацию*. Пустое пространство по-прежнему определяет положение признака в изображении. Когда `MaxPool2D` удаляет часть этих пикселей, он удаляет часть позиционной информации из карты признаков. Это даёт свёрточной сети свойство, называемое **инвариантностью к сдвигу (translation invariance)**. Это означает, что свёрточная сеть с максимальным пулингом будет стремиться не различать признаки по их *расположению* в изображении. («Сдвиг» (translation) — это математический термин для изменения положения объекта без его вращения или изменения формы/размера.)

Посмотрите, что происходит, когда мы многократно применяем максимальный пулинг к следующей карте признаков.

<figure>
<img src="./img/97j8WA1.png" width="800" alt="Пулинг разрушает позиционную информацию.">
</figure>

Две точки на исходном изображении стали неразличимы после многократного пулинга. Другими словами, пулинг уничтожил часть их позиционной информации. Поскольку сеть больше не может различать их на картах признаков, она не может различать их и на исходном изображении: она стала *инвариантной* к этой разнице в положении.

На самом деле пулинг создаёт инвариантность к сдвигу в сети *только на небольших расстояниях*, как в случае с двумя точками на изображении. Признаки, которые изначально находятся далеко друг от друга, останутся различимыми после пулинга: теряется *только часть* позиционной информации, но не вся.

<figure>
<img src="./img/kUMWdcP.png" width="800" alt="Но только на небольших расстояниях. Две точки, расположенные далеко, остаются разделёнными.">
</figure>

Эта инвариантность к небольшим различиям в положении признаков — полезное свойство для классификатора изображений. Из-за различий в перспективе или кадрировании один и тот же тип признака может располагаться в разных частях исходного изображения, но мы хотим, чтобы классификатор распознавал их как одинаковые. Поскольку эта инвариантность *встроена* в сеть, мы можем обойтись гораздо меньшим объёмом данных для обучения: нам больше не нужно учить сеть игнорировать эти различия. Это даёт свёрточным сетям большое преимущество в эффективности по сравнению с сетями, состоящими только из полносвязных слоёв. (Вы увидите ещё один способ получить инвариантность «бесплатно» в **Уроке 6** — **Аугментация данных (Data Augmentation)**!)

# Заключение #

В этом уроке мы узнали о последнем шаге извлечения признаков: **уплотнении** с помощью `MaxPool2D`. В Уроке 4 мы завершим обсуждение свёртки и пулинга, рассмотрев *скользящие окна*.

# Ваша очередь #

А теперь приступайте к [**Упражнению**](https://www.kaggle.com/kernels/fork/11989559), чтобы завершить извлечение признаков, начатое в Уроке 2, увидеть свойство инвариантности в действии, а также узнать о другом виде пулинга — **среднем пулинге (average pooling)**!

---
*Есть вопросы или комментарии? Посетите [форум для обсуждения курса](https://www.kaggle.com/learn/computer-vision/discussion), чтобы пообщаться с другими учащимися.*